# 🕸️ Tablero de Decisión de NO-PATRONES y Grafo de Conocimiento
**Proyecto:** `pipeline-decision-data` · *Hack to the Future 2026 (Córdoba Tech Week)*

Este notebook permite explorar visualmente los resultados del **Grafo de Conocimiento Aumentado Semánticamente (GraphRAG)** construido sobre los 547 registros limpios de la temática ambiental de **Residuos**.

### 🎯 ¿Qué descubrimos?
1. **Vacíos Críticos de Mercado (Solution Gaps):** Problemas ambientales frecuentes en el territorio para los que **no existe oferta ni proyectos reportados** en los medios.
2. **Gemas Ocultas de Innovación:** Tecnologías y modelos circulares atípicos de baja frecuencia con alto potencial disruptivo.
3. **Causas Raíz Cuello de Botella:** Fallas estructurales causantes de múltiples problemas que ninguna solución está atacando.

### 1. Carga de Librerías y Métricas del Grafo

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Image, IFrame

# Definición de rutas relativas seguras
dir_base = Path("../../data/resultados/v1-residuos/no_patrones").resolve()
if not dir_base.exists():
    dir_base = Path("data/resultados/v1-residuos/no_patrones").resolve()

ruta_json = dir_base / "reporte_no_patrones.json"
ruta_umap = dir_base / "mapa_desiertos_2d.png"
ruta_html = dir_base / "grafo_interactivo.html"

with open(ruta_json, "r", encoding="utf-8") as f:
    datos = json.load(f)

res = datos["resumen"]
df_gaps = pd.DataFrame(datos["vacios_solucion"])
df_gemas = pd.DataFrame(datos["gemas_ocultas"])
df_cuellos = pd.DataFrame(datos["causas_cuello_botella"])

# Visualización de KPIs Principales en HTML
kpi_html = f"""
<div style='display: flex; gap: 15px; margin-bottom: 20px;'>
    <div style='background:#1e293b; color:#f8fafc; padding:15px; border-radius:8px; flex:1; border-left:4px solid #ef4444;'>
        <div style='font-size:12px; color:#94a3b8;'>PROBLEMAS ANALIZADOS</div>
        <div style='font-size:24px; font-weight:bold;'>{res['total_problemas']:,}</div>
    </div>
    <div style='background:#1e293b; color:#f8fafc; padding:15px; border-radius:8px; flex:1; border-left:4px solid #10b981;'>
        <div style='font-size:12px; color:#94a3b8;'>SOLUCIONES EN MEDIOS</div>
        <div style='font-size:24px; font-weight:bold;'>{res['total_soluciones']:,}</div>
    </div>
    <div style='background:#1e293b; color:#f8fafc; padding:15px; border-radius:8px; flex:1; border-left:4px solid #8b5cf6;'>
        <div style='font-size:12px; color:#94a3b8;'>NODOS EN EL GRAFO</div>
        <div style='font-size:24px; font-weight:bold;'>{res['total_nodos']:,}</div>
    </div>
    <div style='background:#1e293b; color:#f8fafc; padding:15px; border-radius:8px; flex:1; border-left:4px solid #38bdf8;'>
        <div style='font-size:12px; color:#94a3b8;'>RELACIONES (ARISTAS)</div>
        <div style='font-size:24px; font-weight:bold;'>{res['total_aristas']:,}</div>
    </div>
    <div style='background:#1e293b; color:#f8fafc; padding:15px; border-radius:8px; flex:1; border-left:4px solid #f59e0b;'>
        <div style='font-size:12px; color:#94a3b8;'>RATIO PROBLEMA / SOLUCIÓN</div>
        <div style='font-size:24px; font-weight:bold;'>{res['total_problemas'] / max(res['total_soluciones'], 1):.2f}:1</div>
    </div>
</div>
"""
display(HTML(kpi_html))


### 2. Mapa UMAP 2D: Proyección del Espacio Latente y Desiertos Semánticos
En esta proyección, los puntos **rojos** son problemáticas y los triángulos **verdes** son soluciones.  
Los **círculos amarillos discontinuos** representan las zonas desérticas (No-Patrones): problemas que están completamente aislados y alejados de cualquier solución tecnológica existente.

In [ ]:
if ruta_umap.exists():
    display(Image(filename=str(ruta_umap)))
else:
    print(f"No se encontró la imagen en {ruta_umap}")


### 3. Top Vacíos Críticos de Mercado (Solution Gaps)
Problemas con alta recurrencia cuya distancia matemática a la solución más cercana es máxima ($> 0.80$).

In [ ]:
cols_gaps = ["titulo", "categoria", "causas_raiz", "gap_score", "distancia_solucion_mas_cercana", "solucion_mas_proxima_nombre"]
df_gaps_view = df_gaps[cols_gaps].head(10).copy()
df_gaps_view.columns = ["Problema", "Categoría", "Causas Raíz", "Gap Score", "Distancia a Solución", "Solución más Próxima"]

# Estilizado de tabla con gradiente de color
display(df_gaps_view.style.background_gradient(subset=["Gap Score"], cmap="Reds")\
                          .background_gradient(subset=["Distancia a Solución"], cmap="YlOrRd")\
                          .format({"Gap Score": "{:.2f}", "Distancia a Solución": "{:.4f}"}))


### 4. Top Gemas Ocultas de Innovación (Outliers de Cola Larga)
Iniciativas de baja frecuencia ($n \le 2$) con enfoques tecnológicos singulares que atienden nichos desatendidos.

In [ ]:
top_gemas = df_gemas.head(8).copy()

plt.figure(figsize=(12, 5), facecolor="#0f172a")
ax = plt.gca()
ax.set_facecolor("#0f172a")

y_pos = np.arange(len(top_gemas))
nombres = [str(n)[:35] + "..." for n in top_gemas["nombre"]]

bars = plt.barh(y_pos, top_gemas["score_gema_oculta"], color="#10b981", edgecolor="#34d399", alpha=0.85)
plt.yticks(y_pos, nombres, color="#f8fafc", fontsize=10)
plt.xlabel("Score de Gema Oculta (Singularidad + Potencial de Impacto)", color="#94a3b8", fontsize=11)
plt.title("Gemas Ocultas de Innovación: Tecnologías Atípicas en el Ecosistema", color="#f8fafc", fontsize=13, weight="bold", pad=15)
ax.tick_params(colors="#94a3b8")
for spine in ax.spines.values():
    spine.set_color("#334155")
plt.gca().invert_yaxis()
plt.grid(axis="x", color="#1e293b", linestyle="--", alpha=0.6)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 1.5, bar.get_y() + bar.get_height()/2, f"{w:.1f}", va="center", color="#34d399", weight="bold", fontsize=9)

plt.tight_layout()
plt.show()

display(top_gemas[["nombre", "quien_impulsa", "categoria", "score_gema_oculta", "singularidad_vectorial"]].style.background_gradient(subset=["score_gema_oculta"], cmap="Greens"))


### 5. Causas Raíz Cuello de Botella (Talón de Aquiles)
Causas estructurales que provocan múltiples problemas pero prácticamente ninguna solución registrada las ataca.

In [ ]:
cols_cuellos = ["causa_raiz", "problemas_que_la_padecen", "soluciones_que_la_mitigan", "indice_desatencion", "ejemplo_problema"]
df_cuellos_view = df_cuellos[cols_cuellos].head(10).copy()
df_cuellos_view.columns = ["Causa Raíz", "Problemas que la Padecen", "Soluciones que la Mitigan", "Índice de Desatención", "Caso Representativo"]

display(df_cuellos_view.style.background_gradient(subset=["Índice de Desatención"], cmap="Oranges")\
                            .bar(subset=["Problemas que la Padecen"], color="#f87171")\
                            .format({"Índice de Desatención": "{:.2f}"}))


### 6. Visualizador Interactivo del Grafo Embebido
Puedes hacer zoom con la rueda del ratón, arrastrar nodos y hacer clic en cualquiera de ellos para ver en el panel derecho sus conexiones, causas y grado de desatención.

In [ ]:
if ruta_html.exists():
    display(IFrame(src=str(ruta_html), width="100%", height=650))
else:
    print(f"No se encontró el archivo HTML en {ruta_html}")
